# DeltaH Analysis Notebook (T-MSSC vs Random vs NN-opt)

This notebook computes and visualizes:

1. **(a) `ΔH(W,τ)` heatmaps** for T-MSSC-optimized vs random vs NN-optimized systems (**supports C1, C4**).
2. **(b) `ΔH` spike vs species mass derivative `|dM/dt|` time series** for a selected glider species (**supports C2**).
3. **(c) Trajectory cluster separability at `τ ≈ 3000`** with a **2-cluster fast/slow solution** (**supports C3**).

`ΔH` in section (a) is computed **1:1 with `scripts/clip_deltah_msc_metric.py`**
(same config resolution, same lagged sampling, same null construction, same random-key flow).


In [ ]:
from __future__ import annotations

import os
import sys
import math
import json
import hashlib
import pickle
from pathlib import Path
from types import SimpleNamespace
from typing import Dict, List, Tuple

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from omegaconf import OmegaConf

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.grid"] = True


def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    while True:
        if (cur / "z_deltaH_exampler.py").exists() and (cur / "scripts").exists():
            return cur
        if cur.parent == cur:
            raise FileNotFoundError("Could not find repo root containing z_deltaH_exampler.py")
        cur = cur.parent


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from z_deltaH_exampler import (
    iter_windows,
    sample_velocities_window,
    signature_from_v,
    mean_pairwise_l1,
    make_dirs,
    pairwise_dist_matrix,
    kmedoids_build,
    kmedoids_refine,
    assign_to_medoids,
)
from flow_drift_metrics import infer_log_format, iter_npz_snapshots
from scripts.clip_deltah_msc_metric import resolve_metric_config

print("Repo root:", REPO_ROOT)
print("JAX devices:", jax.devices())


## Configuration

Update paths before running if needed.


In [ ]:
# --- Systems to compare ---
SYSTEMS = {
    "t_mssc": REPO_ROOT / "experiments/opt_msc/checkpoints/test_run/apf_logs",
    "random": REPO_ROOT / "experiments/opt_msc/checkpoints/random_baseline/apf_logs",
    # Adjust this path to your NN-optimized APF logs directory:
    "nn_opt": REPO_ROOT / "experiments/opt_online/checkpoints/test_run/apf_logs",
}

# --- Base optimization config used to resolve metric params exactly as in clip_deltah_msc_metric.py ---
BASE_OPT_CFG = REPO_ROOT / "experiments/opt_msc/optimization/config.yaml"
METRIC_CFG_BY_SYSTEM = {
    "t_mssc": BASE_OPT_CFG,
    "random": BASE_OPT_CFG,
    "nn_opt": BASE_OPT_CFG,
}

# --- Analysis time range in physical steps ---
T1 = 0
T2 = 131_072

# DeltaH heatmap tau values (physical steps)
TAU_PHYS_LIST = [500, 1000, 1500, 2000, 3000, 4000, 6000]

# Keep these explicit for C3 object-collection windows (physical steps)
WINDOW_SIZE_STEPS = 20_000
WINDOW_STEP_STEPS = 20_000

# Projection count used in signature construction for C3
DELTAH_N_PROJ = 16

# C3: two-cluster analysis at tau ~= 3000 physical steps
TAU_PHYS_C3 = 3000
C3_PARTICLES_PER_WINDOW = 96
C3_M_SAMPLES = 64
C3_MAX_OBJECTS_FIT = 2000
C3_SWAP_TRIALS = 250

# C2: system/species selection
SYSTEM_FOR_C2 = "t_mssc"
C2_SPECIES = "fast"  # "fast" or "slow"

# Seed for metric RNG (matches rng_metric role in training metric)
SEED = 123

CACHE_DIR = REPO_ROOT / "experiments/opt_msc/analysis/cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Set True to ignore cache and recompute everything
FORCE_RECOMPUTE = False

for name, path in SYSTEMS.items():
    cfgp = METRIC_CFG_BY_SYSTEM.get(name, BASE_OPT_CFG)
    print(f"{name:8s} -> {path} | exists={path.exists()} | metric_cfg={cfgp}")


In [ ]:
def _save_pickle(path: Path, obj) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump(obj, f)


def _load_pickle(path: Path):
    with open(path, "rb") as f:
        return pickle.load(f)


def _stable_json(obj) -> str:
    return json.dumps(obj, sort_keys=True, ensure_ascii=True, default=str)


def _meta_hash(cache_meta: dict | None) -> str | None:
    if cache_meta is None:
        return None
    s = _stable_json(cache_meta).encode("utf-8")
    return hashlib.sha256(s).hexdigest()


def load_or_compute(cache_path: Path, compute_fn, *, cache_meta: dict | None = None, force: bool = False):
    expected_hash = _meta_hash(cache_meta)

    if cache_path.exists() and not force:
        try:
            raw = _load_pickle(cache_path)
            # New format with metadata-aware cache
            if isinstance(raw, dict) and "_cache_payload" in raw:
                got_hash = raw.get("_cache_meta_hash", None)
                if expected_hash is None or got_hash == expected_hash:
                    return raw["_cache_payload"]
                print(f"[cache invalidated] {cache_path.name}: meta hash changed")
            # Legacy cache payload without metadata: trust only if no cache_meta required
            elif cache_meta is None:
                return raw
            else:
                print(f"[cache invalidated] {cache_path.name}: legacy cache without meta")
        except Exception as e:
            print(f"[cache read failed] {cache_path.name}: {e}; recomputing")

    out = compute_fn()
    payload = {
        "_cache_payload": out,
        "_cache_meta_hash": expected_hash,
        "_cache_meta": cache_meta,
    }
    _save_pickle(cache_path, payload)
    return out


def _cfg_file_hash(cfg_path: Path) -> str:
    txt = cfg_path.read_text() if cfg_path.exists() else ""
    return hashlib.sha256(txt.encode("utf-8")).hexdigest()


def _load_flat_cfg(cfg_path: Path) -> dict:
    cfg = OmegaConf.load(str(cfg_path))
    flat = OmegaConf.merge(
        cfg.get("meta", {}),
        cfg.get("substrate", {}),
        cfg.get("evaluation", {}),
        cfg.get("optimization", {}),
        cfg.get("logging", {}),
        cfg.get("metric", {}),
    )
    return OmegaConf.to_container(flat, resolve=True)


def _load_xy_npz(save_pth: Path, t1: int, t2: int) -> tuple[np.ndarray, np.ndarray]:
    ts = []
    xs = []
    for t, sample in iter_npz_snapshots(str(save_pth), int(t1), int(t2), fields=("lagrangian_xy",)):
        ts.append(int(t))
        xs.append(np.asarray(sample["lagrangian_xy"], dtype=np.float32))
    if not ts:
        raise RuntimeError(f"No lagrangian_xy snapshots found in [{t1},{t2}] for {save_pth}")
    t_arr = np.asarray(ts, dtype=np.int64)
    order = np.argsort(t_arr)
    t_arr = t_arr[order]
    x_arr = np.stack([xs[i] for i in order], axis=0).astype(np.float32)
    return t_arr, x_arr


def _infer_stride_steps(steps: np.ndarray) -> int:
    if steps.size < 2:
        raise RuntimeError("Need at least 2 lagrangian snapshots to infer sample stride.")
    d = np.diff(steps)
    d = d[d > 0]
    if d.size == 0:
        raise RuntimeError("Could not infer positive sample stride from steps.")
    return int(np.median(d))


def _mean_pairwise_l1(sig: jnp.ndarray) -> jnp.ndarray:
    n = sig.shape[0]
    if n < 2:
        return jnp.array(0.0, dtype=sig.dtype)
    d = jnp.mean(jnp.abs(sig[:, None, :] - sig[None, :, :]), axis=2)
    mask = jnp.triu(jnp.ones((n, n), dtype=sig.dtype), k=1)
    denom = jnp.array(n * (n - 1) // 2, dtype=sig.dtype)
    return jnp.sum(d * mask) / jnp.maximum(denom, jnp.array(1.0, dtype=sig.dtype))


def _delta_h_vector_msc_11(xy_np: np.ndarray, cfg_metric: dict, seed: int) -> np.ndarray:
    xy_seq = jnp.asarray(xy_np, dtype=jnp.float32)

    starts = jnp.asarray(cfg_metric["starts"], dtype=jnp.int32)
    W = int(cfg_metric["W"])
    win = int(cfg_metric["window_size_frames"])
    tau = int(cfg_metric["tau_frames"])
    tseg = int(cfg_metric["tseg"])
    m_count = int(cfg_metric["m_count"])
    n_proj = int(cfg_metric["n_proj"])
    null_reps = int(cfg_metric["null_reps"])
    particle_samples = int(cfg_metric["particle_samples"])
    dirs_seed = int(cfg_metric["dirs_seed"])
    periodic = bool(cfg_metric["periodic"])
    domain_y = float(cfg_metric["domain_y"])
    domain_x = float(cfg_metric["domain_x"])

    use_all_lags = (m_count >= tseg)
    base_k_idx = jnp.arange(m_count, dtype=jnp.int32)
    dir_key = jax.random.PRNGKey(dirs_seed)

    def _signature_from_increments(v_s: jnp.ndarray, dirs: jnp.ndarray) -> jnp.ndarray:
        proj = jnp.einsum("msd,ld->msl", v_s, dirs)
        proj = jnp.sort(proj, axis=0)
        sig = jnp.transpose(proj, (1, 2, 0)).reshape(v_s.shape[1], -1)
        return sig

    def _delta_periodic(dx: jnp.ndarray) -> jnp.ndarray:
        if periodic:
            if domain_y > 0:
                dy = (dx[..., 0] + 0.5 * domain_y) % domain_y - 0.5 * domain_y
                dx = dx.at[..., 0].set(dy)
            if domain_x > 0:
                ddx = (dx[..., 1] + 0.5 * domain_x) % domain_x - 0.5 * domain_x
                dx = dx.at[..., 1].set(ddx)
        return dx

    def _delta_h_window(start: jnp.ndarray, key: jax.Array) -> jnp.ndarray:
        key_k, key_p, key_null = jax.random.split(key, 3)
        X_w = jax.lax.dynamic_slice(
            xy_seq,
            (start, 0, 0),
            (win, xy_seq.shape[1], 2),
        )
        n_particles = X_w.shape[1]
        s_count = min(particle_samples, n_particles)

        if use_all_lags:
            k_idx = base_k_idx
        else:
            k_idx = jax.random.choice(key_k, tseg, shape=(m_count,), replace=False)
            k_idx = jnp.sort(k_idx)

        if s_count >= n_particles:
            p_idx = jnp.arange(n_particles, dtype=jnp.int32)
        else:
            p_idx = jax.random.choice(key_p, n_particles, shape=(s_count,), replace=False)
            p_idx = jnp.sort(p_idx)

        X0 = X_w[k_idx][:, p_idx, :]
        X1 = X_w[k_idx + tau][:, p_idx, :]
        dx = _delta_periodic(X1 - X0)

        dt = jnp.maximum(
            jnp.asarray(float(tau) * float(cfg_metric["sample_stride_steps"]), dtype=xy_seq.dtype),
            jnp.asarray(1e-12, dtype=xy_seq.dtype),
        )
        v_s = dx / dt

        dirs = jax.random.normal(dir_key, (n_proj, 2), dtype=xy_seq.dtype)
        dirs = dirs / jnp.maximum(jnp.linalg.norm(dirs, axis=1, keepdims=True), 1e-12)
        sig = _signature_from_increments(v_s, dirs)
        h_real = _mean_pairwise_l1(sig)

        if null_reps <= 0:
            return h_real

        pool = v_s.reshape((-1, 2))
        pool_n = pool.shape[0]

        def _one_null(k: jax.Array) -> jnp.ndarray:
            idx = jax.random.randint(k, (m_count, s_count), 0, pool_n)
            v0 = pool[idx]
            sig0 = _signature_from_increments(v0, dirs)
            return _mean_pairwise_l1(sig0)

        null_keys = jax.random.split(key_null, null_reps)
        h0 = jax.vmap(_one_null)(null_keys)
        return h_real - jnp.median(h0)

    rng_metric = jax.random.PRNGKey(int(seed))
    keys_w = jax.random.split(rng_metric, W)
    h = jax.vmap(_delta_h_window)(starts, keys_w)
    return np.asarray(h, dtype=np.float32)


def _resolve_metric_cfg_for_tau(
    flat_cfg: dict,
    *,
    xy_steps: np.ndarray,
    tau_phys: int,
) -> dict:
    stride = _infer_stride_steps(xy_steps)
    T = int(xy_steps.shape[0])

    # Build args for resolve_metric_config with frame count from logged sequence.
    d = dict(flat_cfg)
    d["rollout_steps"] = int(T * stride)
    d["sample_every_steps"] = int(stride)
    d["time_sampling"] = None
    d["metric_tau_steps"] = int(tau_phys)
    d["metric_tau_frames"] = None

    args = SimpleNamespace(**d)
    cfg_metric = resolve_metric_config(args)
    return cfg_metric


def compute_delta_h_heatmap_msc11(
    save_pth: Path,
    *,
    cfg_path: Path,
    tau_phys_list: List[int],
    t1: int,
    t2: int,
    seed: int,
) -> Dict:
    xy_steps, xy_seq = _load_xy_npz(save_pth, t1=t1, t2=t2)
    flat_cfg = _load_flat_cfg(cfg_path)

    # Use first valid tau as reference for window axis.
    ref_cfg = None
    ref_tau = None
    for tau_phys in tau_phys_list:
        try:
            cfg_m = _resolve_metric_cfg_for_tau(flat_cfg, xy_steps=xy_steps, tau_phys=int(tau_phys))
            ref_cfg = cfg_m
            ref_tau = int(tau_phys)
            break
        except Exception:
            continue
    if ref_cfg is None:
        raise RuntimeError("Could not resolve metric config for any tau in tau_phys_list.")

    W = int(ref_cfg["W"])
    delta_h = np.full((W, len(tau_phys_list)), np.nan, dtype=np.float32)
    valid = np.zeros((W, len(tau_phys_list)), dtype=bool)

    for ti, tau_phys in enumerate(tau_phys_list):
        try:
            cfg_m = _resolve_metric_cfg_for_tau(flat_cfg, xy_steps=xy_steps, tau_phys=int(tau_phys))
            h = _delta_h_vector_msc_11(xy_seq, cfg_m, seed=seed)
            if h.shape[0] != W:
                raise RuntimeError(
                    f"Window count mismatch for tau={tau_phys}: got {h.shape[0]}, expected {W}."
                )
            delta_h[:, ti] = h
            valid[:, ti] = np.isfinite(h)
        except Exception as e:
            print(f"[warn] tau={tau_phys} failed for {save_pth.name}: {e}")

    starts = np.asarray(ref_cfg["starts"], dtype=np.int64)
    stride = int(ref_cfg["sample_stride_steps"])
    win_size_steps = int(ref_cfg["window_size_frames"]) * stride

    window_starts = starts * stride
    window_ends = window_starts + win_size_steps
    window_bounds = np.stack([window_starts, window_ends], axis=1).astype(np.int64)
    window_centers = ((window_starts + window_ends) / 2.0).astype(np.float64)

    return {
        "save_pth": str(save_pth),
        "metric_cfg": str(cfg_path),
        "tau_phys": np.asarray(tau_phys_list, dtype=np.int64),
        "delta_h": delta_h,
        "valid": valid,
        "window_idx": np.arange(W, dtype=np.int64),
        "window_bounds": window_bounds,
        "window_centers": window_centers,
        "sample_stride_steps": stride,
        "ref_tau_phys": ref_tau,
    }


def _tau_frames_from_phys(tau_phys: int, tw: np.ndarray) -> int | None:
    if tw.size < 2:
        return None
    dt = float(np.median(np.diff(tw)))
    if not np.isfinite(dt) or dt <= 0:
        return None
    tau = int(round(float(tau_phys) / dt))
    return max(1, tau)


def collect_tau_objects(
    save_pth: Path,
    *,
    tau_phys: int,
    t1: int,
    t2: int,
    window_size: int,
    window_step: int,
    particles_per_window: int,
    m_samples: int,
    n_proj: int,
    seed: int,
    log_format: str = "auto",
    Lx=None,
    Ly=None,
) -> Dict:
    rng = np.random.default_rng(seed)
    dirs = make_dirs(n_proj, seed=seed + 2000)

    sig_blocks = []
    speed_blocks = []
    obj_window_blocks = []

    window_idx = []
    window_bounds = []

    for local_w, (w_idx, ws, we, tw, Xw) in enumerate(
        iter_windows(str(save_pth), t1, t2, window_size, window_step, log_format=log_format)
    ):
        window_idx.append(int(w_idx))
        window_bounds.append((int(ws), int(we)))

        if tw.size < 2 or Xw is None:
            continue

        tau = _tau_frames_from_phys(int(tau_phys), tw)
        if tau is None or tau >= int(tw.size):
            continue

        Tseg = int(tw.size) - tau
        if Tseg < 4:
            continue

        N = int(Xw.shape[1])
        if N <= 0:
            continue

        p_sel = rng.choice(N, size=min(particles_per_window, N), replace=False)
        m0 = min(m_samples, Tseg)
        k_idx = rng.choice(Tseg, size=m0, replace=False)
        k_idx.sort()

        v_s = sample_velocities_window(
            Xw, tw, tau, k_idx, particle_idx=p_sel, Lx=Lx, Ly=Ly
        )
        if v_s is None:
            continue

        sig = signature_from_v(v_s, dirs)
        speed = np.median(np.linalg.norm(v_s, axis=2), axis=0).astype(np.float32)

        sig_blocks.append(sig.astype(np.float32))
        speed_blocks.append(speed)
        obj_window_blocks.append(np.full(sig.shape[0], local_w, dtype=np.int64))

    if not sig_blocks:
        raise RuntimeError(f"No valid trajectory objects found for tau_phys={tau_phys} in {save_pth}")

    sig_all = np.concatenate(sig_blocks, axis=0)
    speed_all = np.concatenate(speed_blocks, axis=0)
    obj_window_idx = np.concatenate(obj_window_blocks, axis=0)

    return {
        "save_pth": str(save_pth),
        "tau_phys": int(tau_phys),
        "sig": sig_all,
        "speed": speed_all,
        "obj_window_idx": obj_window_idx,
        "window_idx": np.array(window_idx, dtype=np.int64),
        "window_bounds": np.array(window_bounds, dtype=np.int64),
        "window_centers": np.array([(ws + we) / 2.0 for (ws, we) in window_bounds], dtype=np.float64),
    }


def _assign_by_medoid_signatures(sig: np.ndarray, med_sig: np.ndarray):
    dm = np.mean(np.abs(sig[:, None, :] - med_sig[None, :, :]), axis=2)
    order = np.argsort(dm, axis=1)
    labels = order[:, 0].astype(np.int32)
    d1 = dm[np.arange(dm.shape[0]), order[:, 0]]
    d2 = dm[np.arange(dm.shape[0]), order[:, 1]] if dm.shape[1] > 1 else np.full(dm.shape[0], np.inf)
    return labels, d1.astype(np.float32), d2.astype(np.float32), dm


def fit_two_cluster_fast_slow(
    sig: np.ndarray,
    speed: np.ndarray,
    *,
    seed: int,
    max_objects_fit: int,
    swap_trials: int,
) -> Dict:
    rng = np.random.default_rng(seed)
    M = int(sig.shape[0])
    if M < 6:
        raise RuntimeError(f"Need at least 6 objects for stable 2-cluster fit, got {M}")

    if M > max_objects_fit:
        fit_idx = rng.choice(M, size=max_objects_fit, replace=False)
    else:
        fit_idx = np.arange(M)

    sig_fit = sig[fit_idx]
    D_fit = pairwise_dist_matrix(sig_fit, chunk=32)

    med = kmedoids_build(D_fit, 2, rng)
    med, _ = kmedoids_refine(D_fit, med, swap_trials=swap_trials, rng=rng)
    med_sig = sig_fit[med]

    labels_all, d1_all, d2_all, dm_all = _assign_by_medoid_signatures(sig, med_sig)

    speed_median = []
    for k in range(2):
        sk = speed[labels_all == k]
        speed_median.append(float(np.median(sk)) if sk.size else np.nan)

    fast_cluster = int(np.nanargmax(speed_median))
    slow_cluster = 1 - fast_cluster

    gap_abs = float(speed_median[fast_cluster] - speed_median[slow_cluster])
    gap_ratio = float(speed_median[fast_cluster] / (speed_median[slow_cluster] + 1e-12))

    sep_ratio = float(np.median((d2_all - d1_all) / (d1_all + 1e-12)))
    intra_mean = float(np.mean(d1_all))
    inter_medoid = float(np.mean(np.abs(med_sig[0] - med_sig[1])))
    dunn_like = float(inter_medoid / (intra_mean + 1e-12))

    return {
        "labels_all": labels_all,
        "d1_all": d1_all,
        "d2_all": d2_all,
        "med_sig": med_sig,
        "fast_cluster": fast_cluster,
        "slow_cluster": slow_cluster,
        "speed_median_by_cluster": np.array(speed_median, dtype=np.float32),
        "gap_abs": gap_abs,
        "gap_ratio": gap_ratio,
        "sep_ratio": sep_ratio,
        "intra_mean": intra_mean,
        "inter_medoid": inter_medoid,
        "dunn_like": dunn_like,
    }


def load_mass_timeseries_npz(save_pth: Path, t1: int, t2: int) -> Tuple[np.ndarray, np.ndarray]:
    fmt = infer_log_format(str(save_pth))
    if fmt != "npz":
        raise ValueError(f"Expected NPZ APF logs in {save_pth}, got format={fmt}")

    ts = []
    ms = []
    for t, sample in iter_npz_snapshots(str(save_pth), int(t1), int(t2), fields=("A",)):
        A = np.asarray(sample["A"], dtype=np.float32)
        ts.append(int(t))
        ms.append(float(A.sum()))

    if not ts:
        raise RuntimeError(f"No A snapshots found in [{t1}, {t2}] for {save_pth}")

    t_arr = np.asarray(ts, dtype=np.int64)
    m_arr = np.asarray(ms, dtype=np.float64)
    order = np.argsort(t_arr)
    return t_arr[order], m_arr[order]


def aggregate_mass_by_windows(
    t_mass: np.ndarray,
    mass: np.ndarray,
    window_bounds: np.ndarray,
) -> np.ndarray:
    out = np.full(window_bounds.shape[0], np.nan, dtype=np.float64)
    for i, (ws, we) in enumerate(window_bounds):
        mask = (t_mass >= int(ws)) & (t_mass < int(we))
        if np.any(mask):
            out[i] = float(np.mean(mass[mask]))
    return out


def zscore_nan(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    mu = np.nanmean(x)
    sd = np.nanstd(x)
    if not np.isfinite(sd) or sd <= 0:
        return np.full_like(x, np.nan)
    return (x - mu) / sd


def pca_2d(x: np.ndarray) -> np.ndarray:
    x0 = x - np.mean(x, axis=0, keepdims=True)
    _, _, vh = np.linalg.svd(x0, full_matrices=False)
    return x0 @ vh[:2].T


## (a) Compute `ΔH(W,τ)` for all systems (supports C1, C4)


In [ ]:
deltah_by_system = {}

for name, path in SYSTEMS.items():
    if not path.exists():
        print(f"[skip] {name}: missing path {path}")
        continue

    cfg_path = METRIC_CFG_BY_SYSTEM.get(name, BASE_OPT_CFG)
    if not cfg_path.exists():
        print(f"[skip] {name}: missing metric cfg {cfg_path}")
        continue

    cache_file = CACHE_DIR / f"deltah_msc11_{name}.pkl"
    cache_meta = {
        "kind": "deltah_msc11",
        "system": name,
        "save_pth": str(path),
        "metric_cfg_path": str(cfg_path),
        "metric_cfg_sha256": _cfg_file_hash(cfg_path),
        "tau_phys_list": [int(x) for x in TAU_PHYS_LIST],
        "t1": int(T1),
        "t2": int(T2),
        "seed": int(SEED),
    }

    def _compute():
        return compute_delta_h_heatmap_msc11(
            path,
            cfg_path=cfg_path,
            tau_phys_list=TAU_PHYS_LIST,
            t1=T1,
            t2=T2,
            seed=SEED,
        )

    deltah_by_system[name] = load_or_compute(
        cache_file,
        _compute,
        cache_meta=cache_meta,
        force=FORCE_RECOMPUTE,
    )
    print(
        f"[{name}] windows={deltah_by_system[name]['delta_h'].shape[0]}, "
        f"taus={deltah_by_system[name]['delta_h'].shape[1]}, "
        f"stride={deltah_by_system[name]['sample_stride_steps']}"
    )


In [ ]:
if not deltah_by_system:
    raise RuntimeError("No systems available. Check SYSTEMS paths.")

all_vals = []
for res in deltah_by_system.values():
    v = res["delta_h"]
    all_vals.append(v[np.isfinite(v)])
all_vals = np.concatenate([v for v in all_vals if v.size > 0]) if all_vals else np.array([0.0])

vmax = np.nanpercentile(np.abs(all_vals), 98) if all_vals.size else 1.0
vmax = float(max(vmax, 1e-6))

n = len(deltah_by_system)
fig, axes = plt.subplots(1, n, figsize=(5.5 * n, 4.8), sharey=True)
if n == 1:
    axes = [axes]

for ax, (name, res) in zip(axes, deltah_by_system.items()):
    im = ax.imshow(
        res["delta_h"],
        origin="lower",
        aspect="auto",
        cmap="coolwarm",
        vmin=-vmax,
        vmax=vmax,
    )
    ax.set_title(f"{name}: ΔH(W,τ)")
    ax.set_xlabel("tau (physical steps)")
    ax.set_xticks(np.arange(len(res["tau_phys"])))
    ax.set_xticklabels([str(int(x)) for x in res["tau_phys"]], rotation=45, ha="right")
    ax.set_ylabel("window index")

fig.colorbar(im, ax=axes, shrink=0.85, label="ΔH")
fig.suptitle("(a) ΔH heatmaps: T-MSSC vs random vs NN-opt", y=1.02)
plt.tight_layout()
plt.show()


## (c) Trajectory cluster separability at `τ ≈ 3000` (supports C3)

We build per-particle trajectory signatures at `tau_phys = TAU_PHYS_C3`, fit a **2-medoid** model,
and report fast/slow speed gap and separability metrics.


In [ ]:
c3_by_system = {}

for name, path in SYSTEMS.items():
    if not path.exists():
        continue

    cache_file = CACHE_DIR / f"c3_tau{TAU_PHYS_C3}_{name}.pkl"
    cache_meta = {
        "kind": "c3_two_cluster",
        "system": name,
        "save_pth": str(path),
        "tau_phys": int(TAU_PHYS_C3),
        "t1": int(T1),
        "t2": int(T2),
        "window_size_steps": int(WINDOW_SIZE_STEPS),
        "window_step_steps": int(WINDOW_STEP_STEPS),
        "particles_per_window": int(C3_PARTICLES_PER_WINDOW),
        "m_samples": int(C3_M_SAMPLES),
        "n_proj": int(DELTAH_N_PROJ),
        "seed": int(SEED),
    }

    def _compute():
        objs = collect_tau_objects(
            path,
            tau_phys=TAU_PHYS_C3,
            t1=T1,
            t2=T2,
            window_size=WINDOW_SIZE_STEPS,
            window_step=WINDOW_STEP_STEPS,
            particles_per_window=C3_PARTICLES_PER_WINDOW,
            m_samples=C3_M_SAMPLES,
            n_proj=DELTAH_N_PROJ,
            seed=SEED + 17,
            log_format="auto",
            Lx=None,
            Ly=None,
        )
        fit = fit_two_cluster_fast_slow(
            objs["sig"],
            objs["speed"],
            seed=SEED + 33,
            max_objects_fit=C3_MAX_OBJECTS_FIT,
            swap_trials=C3_SWAP_TRIALS,
        )
        out = dict(objs)
        out.update(fit)
        return out

    c3_by_system[name] = load_or_compute(
        cache_file,
        _compute,
        cache_meta=cache_meta,
        force=FORCE_RECOMPUTE,
    )

summary_rows = []
for name, res in c3_by_system.items():
    summary_rows.append(
        (
            name,
            int(res["sig"].shape[0]),
            float(res["sep_ratio"]),
            float(res["dunn_like"]),
            float(res["gap_ratio"]),
            float(res["gap_abs"]),
            int(res["fast_cluster"]),
            int(res["slow_cluster"]),
        )
    )

summary_rows = sorted(summary_rows, key=lambda x: x[0])
print("name      n_obj   sep_ratio   dunn_like   gap_ratio   gap_abs   fast  slow")
for r in summary_rows:
    print(f"{r[0]:8s} {r[1]:6d}  {r[2]:9.4f}  {r[3]:9.4f}  {r[4]:9.4f}  {r[5]:8.4f}   {r[6]:4d}  {r[7]:4d}")


In [ ]:
if not c3_by_system:
    raise RuntimeError("No C3 results. Check system paths and tau/window settings.")

names = list(c3_by_system.keys())
sep_vals = [float(c3_by_system[n]["sep_ratio"]) for n in names]
dunn_vals = [float(c3_by_system[n]["dunn_like"]) for n in names]
gap_vals = [float(c3_by_system[n]["gap_ratio"]) for n in names]

x = np.arange(len(names))
width = 0.25

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(x - width, sep_vals, width=width, label="sep_ratio")
ax.bar(x, dunn_vals, width=width, label="dunn_like")
ax.bar(x + width, gap_vals, width=width, label="fast/slow gap ratio")
ax.set_xticks(x)
ax.set_xticklabels(names)
ax.set_title(f"(c) C3 metrics at tau_phys={TAU_PHYS_C3}")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Optional: 2D PCA view of trajectory signatures per system (colored by fast/slow cluster)
MAX_POINTS_PER_SYSTEM = 1200
rng = np.random.default_rng(SEED + 999)

n = len(c3_by_system)
fig, axes = plt.subplots(1, n, figsize=(5.2 * n, 4.5))
if n == 1:
    axes = [axes]

for ax, (name, res) in zip(axes, c3_by_system.items()):
    sig = res["sig"]
    labels = res["labels_all"]
    M = sig.shape[0]
    if M > MAX_POINTS_PER_SYSTEM:
        idx = rng.choice(M, size=MAX_POINTS_PER_SYSTEM, replace=False)
    else:
        idx = np.arange(M)

    pcs = pca_2d(sig[idx])
    lbl = labels[idx]

    slow = int(res["slow_cluster"])
    fast = int(res["fast_cluster"])
    ax.scatter(pcs[lbl == slow, 0], pcs[lbl == slow, 1], s=7, alpha=0.6, label=f"slow={slow}")
    ax.scatter(pcs[lbl == fast, 0], pcs[lbl == fast, 1], s=7, alpha=0.6, label=f"fast={fast}")
    ax.set_title(name)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.legend(loc="best")

fig.suptitle("(c) Two-cluster fast/slow structure at tau≈3000", y=1.02)
plt.tight_layout()
plt.show()


## (b) `ΔH` spikes vs selected species mass derivative `|dM/dt|` (supports C2)

Species mass is estimated as:

- `M_species(window) = f_species(window) * mean_total_mass(window)`
- where `f_species(window)` is the fraction of trajectory objects assigned to the chosen species (fast/slow cluster).


In [ ]:
if SYSTEM_FOR_C2 not in deltah_by_system:
    raise RuntimeError(f"SYSTEM_FOR_C2={SYSTEM_FOR_C2!r} missing in deltah_by_system")
if SYSTEM_FOR_C2 not in c3_by_system:
    raise RuntimeError(f"SYSTEM_FOR_C2={SYSTEM_FOR_C2!r} missing in c3_by_system")

res_h = deltah_by_system[SYSTEM_FOR_C2]
res_c3 = c3_by_system[SYSTEM_FOR_C2]
sys_path = Path(res_h["save_pth"])

# pick nearest tau column to TAU_PHYS_C3 for deltaH spike analysis
tau_phys = np.asarray(res_h["tau_phys"])
tau_col = int(np.argmin(np.abs(tau_phys - TAU_PHYS_C3)))
selected_tau_phys = int(tau_phys[tau_col])

delta_h_series = np.asarray(res_h["delta_h"][:, tau_col], dtype=np.float64)
window_centers = np.asarray(res_h["window_centers"], dtype=np.float64)
window_bounds = np.asarray(res_h["window_bounds"], dtype=np.int64)

# species fraction per window from C3 labels
labels_all = np.asarray(res_c3["labels_all"], dtype=np.int32)
obj_window_idx = np.asarray(res_c3["obj_window_idx"], dtype=np.int64)
if C2_SPECIES == "fast":
    species_cluster = int(res_c3["fast_cluster"])
elif C2_SPECIES == "slow":
    species_cluster = int(res_c3["slow_cluster"])
else:
    raise ValueError("C2_SPECIES must be 'fast' or 'slow'")

species_frac = np.full(window_bounds.shape[0], np.nan, dtype=np.float64)
for w in range(window_bounds.shape[0]):
    m = obj_window_idx == w
    if np.any(m):
        species_frac[w] = float(np.mean(labels_all[m] == species_cluster))

# total mass timeseries from A logs
mass_t, mass = load_mass_timeseries_npz(sys_path, T1, T2)
mass_win = aggregate_mass_by_windows(mass_t, mass, window_bounds)

species_mass = species_frac * mass_win
d_species = np.abs(np.gradient(species_mass, window_centers))

z_dh = zscore_nan(delta_h_series)
z_dm = zscore_nan(d_species)

mask = np.isfinite(z_dh) & np.isfinite(z_dm)
corr = float(np.corrcoef(z_dh[mask], z_dm[mask])[0, 1]) if np.sum(mask) >= 3 else np.nan

print(f"System: {SYSTEM_FOR_C2}")
print(f"Selected species cluster: {species_cluster} ({C2_SPECIES})")
print(f"DeltaH tau used (physical): {selected_tau_phys}")
print(f"Correlation(z(DeltaH), z(|dM_species/dt|)): {corr:.4f}")


In [ ]:
# Time-series overlay with spike markers
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(window_centers, z_dh, marker="o", label=f"z(DeltaH), tau={selected_tau_phys}")
ax.plot(window_centers, z_dm, marker="s", label=f"z(|dM_species/dt|), species={C2_SPECIES}")

if np.isfinite(delta_h_series).any():
    thr = np.nanquantile(delta_h_series, 0.9)
    spike_mask = delta_h_series >= thr
    ax.scatter(window_centers[spike_mask], z_dh[spike_mask], s=55, marker="*", label="DeltaH spikes (90th pct)")

ax.set_title(f"(b) DeltaH spikes vs species mass derivative |dM/dt| ({SYSTEM_FOR_C2})")
ax.set_xlabel("Physical step (window center)")
ax.set_ylabel("z-score")
ax.legend(loc="best")
plt.tight_layout()
plt.show()


In [ ]:
# Optional scatter view for C2 relationship
fig, ax = plt.subplots(figsize=(5.5, 5.0))
mask = np.isfinite(z_dh) & np.isfinite(z_dm)
ax.scatter(z_dh[mask], z_dm[mask], s=35, alpha=0.8)
ax.set_xlabel("z(DeltaH)")
ax.set_ylabel("z(|dM_species/dt|)")
ax.set_title(f"(b) C2 scatter, corr={corr:.3f}")
plt.tight_layout()
plt.show()


## Notes

- If `nn_opt` path does not exist, it is skipped automatically.
- All expensive computations are cached in `experiments/opt_msc/analysis/cache`.
- For strict reproducibility, keep `SEED` fixed.
- If you need closer alignment with a previous dist-pipeline run, match `DELTAH_*` and `C3_*` params to that run.
